# NYC 311 Borough Risk Heatmap Generator

**Purpose:** Generate an interactive choropleth heatmap of NYC borough-level complaint risk scores.

**Pipeline:**
- Load `civic_lens.output.nyc_borough_risk` (borough × complaint_type risk scores)
- Aggregate to borough-level weighted risk scores
- Join with NYC borough GeoJSON boundaries
- Render Folium choropleth with interactive popups

**Output:** `viz/output/nyc_heatmap.html` (interactive map)

**✅ Data Quality Fix Applied:**  
The pipeline now uses **model v2** (trained only on never_resolved=0) and scores representative complaints:  
- Previous issue: Model trained on resolved complaints but scored on never_resolved=1 edge cases
- Result: Predictions inflated to ~640 days (2.5X too high)
- Fix: Retrained model v2 on never_resolved=0 only, score on same distribution
- Current predictions: **~305 days average** (realistic and aligned with ground truth)

**Data Flow:**
```
civic_lens.output.nyc_borough_risk (401 rows)
    ↓
Aggregate by borough (5 boroughs)
    ↓
Join with tmp_nyc_boroughs.geojson
    ↓
Folium choropleth visualization
    ↓
nyc_heatmap.html
```

In [0]:
%pip install geopandas folium -q

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
import os
import pandas as pd
import geopandas as gpd
import folium
from pyspark.sql import SparkSession

# Add src directory to path for geo_utils
sys.path.insert(0, '/Workspace/Users/pawanvirat32@gmail.com/civic-lens/src')
from geo_utils import normalize_borough

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

print("✓ Libraries loaded successfully")

In [0]:
print("=" * 70)
print("NYC 311 BOROUGH RISK HEATMAP GENERATOR")
print("=" * 70)
print("\n[1/5] Loading borough risk data from Databricks...")

# Load the risk table (using ML schema with latest model v2 predictions)
risk_df = spark.table("civic_lens.ml.nyc_borough_scores").toPandas()

print(f"   ✓ Loaded {len(risk_df):,} borough-complaint_type combinations")
print(f"   ✓ Columns: {list(risk_df.columns)}")

# Display sample
display(risk_df.head())

In [0]:
print("\n[2/5] Aggregating risk scores by borough...")

# For the heatmap, we want ONE risk score per borough (not per complaint_type)
# Strategy: Weight by complaint_count, use composite_risk_score
borough_agg = risk_df.groupby('borough').agg({
    'complaint_count': 'sum',
    'composite_risk_score': lambda x: (x * risk_df.loc[x.index, 'complaint_count']).sum() / risk_df.loc[x.index, 'complaint_count'].sum(),
    'avg_predicted_days': 'mean',
    'risk_tier': lambda x: x.mode()[0] if not x.empty else 'LOW'
}).reset_index()

# Rename columns for clarity
borough_agg.columns = ['borough', 'total_complaints', 'weighted_risk_score', 'avg_days', 'risk_tier']

# Get top 5 complaint types per borough
top_complaints_by_borough = {}
for borough in risk_df['borough'].unique():
    borough_data = risk_df[risk_df['borough'] == borough].nlargest(5, 'complaint_count')
    top_complaints_by_borough[borough] = borough_data[['complaint_type', 'complaint_count', 'avg_predicted_days']].to_dict('records')

borough_agg['top_complaints'] = borough_agg['borough'].map(top_complaints_by_borough)

print(f"   ✓ Aggregated to {len(borough_agg)} boroughs")
print(f"\n   Borough Risk Summary:")
for _, row in borough_agg.sort_values('weighted_risk_score', ascending=False).iterrows():
    print(f"      {row['borough']:15} | Score: {row['weighted_risk_score']:.3f} | Tier: {row['risk_tier']:6} | {row['total_complaints']:,} complaints")

display(borough_agg)

In [0]:
print("\n[3/5] Loading NYC borough GeoJSON boundaries...")

geojson_path = '/Workspace/Users/pawanvirat32@gmail.com/civic-lens/src/tmp_nyc_boroughs.geojson'
gdf = gpd.read_file(geojson_path)

print(f"   ✓ Loaded {len(gdf)} borough geometries")
print(f"   ✓ GeoJSON borough names: {sorted(gdf['BoroName'].tolist())}")

display(gdf[['BoroName', 'BoroCode', 'Shape_Area']].head())

In [0]:
print("\n[4/5] Merging risk data with GeoJSON...")

# Normalize borough names in both datasets
borough_agg['borough_norm'] = borough_agg['borough'].str.strip().str.lower()
gdf['BoroName_norm'] = gdf['BoroName'].str.strip().str.lower()

# Merge
gdf_merged = gdf.merge(
    borough_agg,
    left_on='BoroName_norm',
    right_on='borough_norm',
    how='left'
)

# Check merge success
matched = gdf_merged['weighted_risk_score'].notna().sum()
print(f"   ✓ Matched {matched}/{len(gdf)} boroughs successfully")

if matched < len(gdf):
    unmatched = gdf_merged[gdf_merged['weighted_risk_score'].isna()]['BoroName'].tolist()
    print(f"   ⚠️  Unmatched boroughs: {unmatched}")

# Fill missing values with 0 for unmapped boroughs
gdf_merged['weighted_risk_score'] = gdf_merged['weighted_risk_score'].fillna(0)
gdf_merged['total_complaints'] = gdf_merged['total_complaints'].fillna(0).astype(int)
gdf_merged['risk_tier'] = gdf_merged['risk_tier'].fillna('LOW')

print(f"\n   Merged data preview:")
display(gdf_merged[['BoroName', 'weighted_risk_score', 'total_complaints', 'risk_tier']])

In [0]:
print("\n[5/5] Creating Folium choropleth heatmap...")

# Calculate map center (NYC coordinates)
nyc_center = [40.7128, -74.0060]  # Manhattan coordinates

# Create base map
m = folium.Map(
    location=nyc_center,
    zoom_start=10,
    tiles='CartoDB positron'
)

# Create choropleth layer
folium.Choropleth(
    geo_data=gdf_merged.to_json(),
    name='Borough Risk',
    data=gdf_merged,
    columns=['BoroName', 'weighted_risk_score'],
    key_on='feature.properties.BoroName',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Composite Risk Score',
    nan_fill_color='lightgray'
).add_to(m)

# Add interactive popups with detailed info
for _, row in gdf_merged.iterrows():
    # Build top complaints HTML
    top_complaints_html = "<br><b>Top 5 Complaint Types:</b><br>"
    top_complaints = row['top_complaints']
    if isinstance(top_complaints, list) and len(top_complaints) > 0:
        for i, complaint in enumerate(top_complaints, 1):
            complaint_type = complaint['complaint_type']
            count = int(complaint['complaint_count'])
            days = int(complaint['avg_predicted_days'])
            top_complaints_html += f"<span style='font-size:11px'>{i}. <b>{complaint_type}</b> ({count:,} complaints, ~{days} days)</span><br>"
    else:
        top_complaints_html += "<i>No data</i><br>"
    
    folium.GeoJson(
        row.geometry,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': 'transparent',
            'weight': 0
        },
        tooltip=folium.Tooltip(
            f"""
            <b>{row['BoroName']}</b><br>
            <hr style='margin:4px 0'>
            Composite Risk Score: <b>{row['weighted_risk_score']:.3f}</b><br>
            Risk Tier: <b>{row['risk_tier']}</b><br>
            Total Complaints: <b>{int(row['total_complaints']):,}</b><br>
            Avg Resolution: <b>{int(row['avg_days'])} days</b> (model v2)<br>
            {top_complaints_html}
            """,
            style='font-size: 12px; font-family: Arial;'
        )
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

print(f"   ✓ Choropleth layer created")
print(f"   ✓ Interactive popups added for {len(gdf_merged)} boroughs")

In [0]:
# Save to output directory
output_path = '/Workspace/Users/pawanvirat32@gmail.com/civic-lens/viz/output/nyc_heatmap.html'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
m.save(output_path)

print(f"\n   ✓ Heatmap saved to: {output_path}")
print(f"   ✓ Map centered at: {nyc_center}")
print(f"   ✓ Zoom level: 10")

# Print summary statistics
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Total boroughs: {len(gdf_merged)}")
print(f"Total complaints analyzed: {int(gdf_merged['total_complaints'].sum()):,}")
print(f"Risk score range: {gdf_merged['weighted_risk_score'].min():.3f} - {gdf_merged['weighted_risk_score'].max():.3f}")
print(f"\nRisk tier distribution:")
for tier in ['HIGH', 'MEDIUM', 'LOW']:
    count = len(gdf_merged[gdf_merged['risk_tier'] == tier])
    if count > 0:
        print(f"  {tier}: {count} boroughs")
print("\n" + "=" * 70)
print("✅ NYC 311 Borough Risk Heatmap generation complete!")
print("=" * 70)

# Display the map inline
m